# 02 — Обучение модели
> **v2 | Агент удержания полосы + ПИД-регулятор | ветка: `v2-agent-pid`**

**Время:** ~10–15 минут на T4  
Загружаем из кэша (мгновенно), обучаем LaneCNN, сохраняем `models/model_v2.pth`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os

# Явно переходим в папку проекта
REPO_PATH = '/content/drive/MyDrive/diploma'
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.dataset_v2 import load_and_cache, get_loaders
from src.model_v2   import build_model
from src.train_v2   import train, evaluate

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство:', DEVICE)

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# Загружаем кэш — мгновенно, всё в RAM
images, angles = load_and_cache(
    csv_path=None,          # не нужен, кэш уже есть
    images_dir=None,
    cache_path='data/cache.npy'
)
print(f'Загружено: {len(images)} сэмплов')

In [ ]:
# DataLoader: batch_size=128, num_workers=2, pin_memory=True
train_loader, val_loader, test_loader = get_loaders(
    images, angles, batch_size=128
)
print(f'Train батчей: {len(train_loader)}  '
      f'Val: {len(val_loader)}  Test: {len(test_loader)}')

In [ ]:
# Строим модель
model = build_model(DEVICE)

In [ ]:
# Обучаем (20 эпох, EarlyStopping patience=5)
history = train(
    model, train_loader, val_loader,
    epochs=20,
    lr=3e-4,
    patience=5,
    model_path='models/model_v2.pth',
    device=DEVICE
)

In [ ]:
# ── Кривые обучения ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history['train_loss'], label='Train loss')
ax.plot(history['val_loss'],   label='Val loss')
ax.set_xlabel('Эпоха')
ax.set_ylabel('MSE Loss')
ax.set_title('Кривые обучения LaneCNN')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/training_curves.png', dpi=120)
plt.show()

In [ ]:
# ── Метрики на тестовой выборке ───────────────────────────────────
# Загружаем лучшую модель
model.load_state_dict(torch.load('models/model_v2.pth', map_location=DEVICE))
metrics = evaluate(model, test_loader, DEVICE)

print('\n── Метрики на тесте ──────────────────')
print(f'  MSE:      {metrics["mse"]:.5f}')
print(f'  MAE:      {metrics["mae"]:.5f}')
print(f'  RMSE:     {metrics["rmse"]:.5f}')
print(f'  Accuracy: {metrics["accuracy"]*100:.1f}%  (в/вне полосы)')

In [ ]:
# ── Scatter: предсказание vs реальное значение ────────────────────
model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for imgs, ang in test_loader:
        pred = model(imgs.to(DEVICE)).cpu().numpy().flatten()
        all_pred.extend(pred)
        all_true.extend(ang.numpy().flatten())

all_pred = np.array(all_pred)
all_true = np.array(all_true)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(all_true, all_pred, alpha=0.3, s=10)
ax.plot([-1, 1], [-1, 1], 'r--', linewidth=1.5, label='Идеал')
ax.axvline(-0.15, color='gray', linestyle=':', alpha=0.7)
ax.axvline( 0.15, color='gray', linestyle=':', alpha=0.7, label='Граница ±0.15')
ax.set_xlabel('Реальный угол')
ax.set_ylabel('Предсказанный угол')
ax.set_title('Предсказание vs Реальное')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/scatter.png', dpi=120)
plt.show()